# 06 · The Right job and the report

**Run first:**
```bash
uv run pytest -v tests/unit/dtrack_right/test_job.py tests/unit/dtrack_right/report/
just test e2e
DTRACK_LIVE_KEEP=1 just live test test_job_flow.py
```
In the Databricks UI the live run is under **Jobs & Pipelines → Runs** (it is a one-time
`SUBMIT_RUN`), not under the Jobs list.

**Read:** `src/dtrack_right/job.py::run` and `src/dtrack_right/report/build.py::_summary`.

In [ ]:
import json

import nbkit
import pandas as pd
from nbkit import code, table, tree

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1 · The job starts before any data exists

`job.run` polls: it resolves the run folder, reconciles every new `.left.json`, drains units,
and exits by itself after `idle_timeout` with nothing to do. That is why the test submits the
job first and runs the Left side afterwards, the same as production.

In [ ]:
from dtrack_right.report import build_report

folder = nbkit.sandbox("06")
data, store = nbkit.stage_one(folder)
published, results = nbkit.stage_two(folder, data, store)

cases = []


def status(label):
    summary = json.loads(build_report(store, folder / "report")["summary.json"].read_text())
    cases.append(
        {
            "case": label,
            "status": summary["status"],
            "units": f"{summary['units_completed']}/{summary['units_expected']}",
            **summary["findings"],
        }
    )


status("complete run")
table(pd.DataFrame(cases))

## 2 · What makes a run `PARTIAL`

```python
if dead or (expected and completed < expected) or not state["closed"]:
    status = "PARTIAL"
```
`closed` is `manifest/_CLOSED`, written by the Left `col-check` when it has published every
unit. Findings (count diffs, missing partitions, column diffs) never change the status.

**Common mistake:** forgetting the third condition. Without `manifest/_CLOSED` the Right side
cannot know the Left side is finished, even if every result it has seen is complete.

In [ ]:
closed = store.get_json("manifest/_CLOSED")
(store.root / "manifest/_CLOSED").unlink()
status("without manifest/_CLOSED")
store.put_json("manifest/_CLOSED", closed)

result_key = store.list_keys("results/")[0]
saved = store.get_json(result_key)
(store.root / result_key).unlink()
status("one result missing")
store.put_json(result_key, saved)

unit_id = result_key.removeprefix("results/").removesuffix(".json")
(store.root / result_key).unlink()
store.put_dead(unit_id, {"unit_id": unit_id, "error": "SparkException after 3 attempts"})
status("one unit dead instead of compared")
(store.root / f"dead/{unit_id}.json").unlink()
store.put_json(result_key, saved)
status("restored")
table(pd.DataFrame(cases[1:]))

## 3 · The report header

`Latest run: <local time with EDT/EST> | RUN <run id>` and one summary line. The separator,
status colours and time zone are module constants in `report/html.py` (`SEP`, `STATUS_COLORS`,
`REPORT_ZONE`).

In [ ]:
import re

html = (folder / "report" / "index.html").read_text()
code(
    " ".join(
        re.sub("<[^>]+>", " ", re.search("Latest run:.*?</div>", html, re.DOTALL).group(0)).split()
    ),
    label="subtitle",
)
code(
    " ".join(re.sub("<[^>]+>", " ", re.search("STATUS=.*?</p>", html).group(0)).split()),
    label="summary line",
)

## 4 · Publishing: history first, pointer last

`publish_report` writes `history/<run>/…` and then `latest/pointer.json`; `fetch-report`
follows the pointer. The pointer never names an incomplete report, the same rule again.

In [ ]:
from dtrack_left.backends.store import fetch_report
from dtrack_right.report import publish_report

report_store = nbkit.VolumeStore(folder / "report_store")
files = build_report(store, folder / "report")
publish_report(report_store, "nb", files, json.loads(files["summary.json"].read_text()))
tree(report_store.root)
print(fetch_report(report_store, folder / "fetched")["run_id"])